# Experiment 3 — the alternatives on the same data

The same join as Experiment 1, first with AQE's skew handling switched back on, then with a broadcast.

In [1]:
import time
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("m5-exp3-alternatives").getOrCreate()
fact = spark.read.parquet("/opt/data/m5/orders_skew")
dim = spark.read.parquet("/opt/data/m4/customers_big")

def timed(label, df):
    t0 = time.perf_counter()
    df.write.format("noop").mode("overwrite").save()
    print(f"{label}: {time.perf_counter() - t0:.1f}s")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/01 07:04:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## A — AQE skew join on (the default)

AQE treats a shuffle partition as skewed when it is larger than **both** `spark.sql.adaptive.skewJoin.skewedPartitionFactor` (5) × the median partition **and** `spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes` (256 MB).

**PREDICT:** from Experiment 1's shuffle-read numbers, will the hot partition qualify? If it does, what happens to the join stage's task count and its max task duration? Runtime relative to Experiment 1 and to your salted join.

In [2]:
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
aqe = fact.join(dim, "customer_id")
timed("AQE skew join", aqe)

[Stage 6:=======================================>                 (13 + 6) / 19]

AQE skew join: 5.9s


In [3]:
timed("AQE skew join (warm)", aqe)

[Stage 11:==================================================>     (17 + 2) / 19]

AQE skew join (warm): 2.6s


Check in the SQL / DataFrame tab whether it fired: the final plan's `AQEShuffleRead` shows *number of skewed partitions* and *number of skewed partition splits*.

## B — broadcast the dimension

`customers_big` is ~149 MiB on disk, 2 M rows.

**PREDICT:** does the hot key still produce a slow task? Why or why not? What is the cost of this variant, and where does it land? Runtime relative to the others.

In [4]:
bc = fact.join(F.broadcast(dim), "customer_id")
bc.explain("formatted")
timed("broadcast join", bc)

== Physical Plan ==
AdaptiveSparkPlan (8)
+- Project (7)
   +- BroadcastHashJoin Inner BuildRight (6)
      :- Filter (2)
      :  +- Scan parquet  (1)
      +- BroadcastExchange (5)
         +- Filter (4)
            +- Scan parquet  (3)


(1) Scan parquet 
Output [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Batched: true
Location: InMemoryFileIndex [file:/opt/data/m5/orders_skew]
PushedFilters: [IsNotNull(customer_id)]
ReadSchema: struct<order_id:bigint,customer_id:int,shop_id:int,amount:double,order_date:date,note:string>

(2) Filter
Input [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Condition : isnotnull(customer_id#1)

(3) Scan parquet 
Output [5]: [customer_id#6, tier#7, country#8, profile_a#9, profile_b#10]
Batched: true
Location: InMemoryFileIndex [file:/opt/data/m4/customers_big]
PushedFilters: [IsNotNull(customer_id)]
ReadSchema: struct<customer_id:int,tier:string,country:string,profile_a:string,profile_b:string>

[Stage 13:>                                                         (0 + 8) / 8]

broadcast join: 3.3s


## Stop

In [ ]:
spark.stop()